<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>📅 Chapter 19 — Time Series Analysis</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 75 mins | Level: Intermediate</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Decompose a time series into trend, seasonality, and noise
- Test for stationarity and apply differencing
- Apply rolling statistics for smoothing and feature engineering
- Build a simple ARIMA forecast
- Use ML-based approaches (lag features) for time series prediction

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.seasonal   import seasonal_decompose
from statsmodels.tsa.stattools  import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.linear_model       import LinearRegression
from sklearn.metrics            import mean_absolute_error, mean_squared_error

%matplotlib inline
np.random.seed(42)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Synthetic: Indian E-commerce Daily Orders (3 years)
# ─────────────────────────────────────────────────────────────
dates = pd.date_range('2021-01-01', '2023-12-31', freq='D')
n     = len(dates)

trend      = np.linspace(1000, 2500, n)
weekly_s   = 200 * np.sin(2 * np.pi * np.arange(n) / 7)
annual_s   = 400 * np.sin(2 * np.pi * np.arange(n) / 365 - np.pi/2)

# Festival spikes: Diwali (Oct-Nov), New Year, Republic Day
festival   = np.zeros(n)
for dt_str in ['2021-11-04', '2022-10-24', '2023-11-12',   # Diwali
               '2022-01-01', '2023-01-01',                  # New Year
               '2021-08-15', '2022-08-15', '2023-08-15']:   # Independence Day
    idx = (dates == dt_str)
    festival[idx] = 800

noise = np.random.normal(0, 80, n)
orders = (trend + weekly_s + annual_s + festival + noise).clip(min=100).round()

ts = pd.Series(orders, index=dates, name='orders')
print(f'Time series: {ts.index[0].date()} to {ts.index[-1].date()}, {len(ts)} days')

ts.plot(figsize=(14, 4), title='Flipkart-style Daily Orders (Synthetic)', color='steelblue')
plt.ylabel('Orders')
plt.show()

## 📖 Section A — Decomposition and Stationarity

```python
from statsmodels.tsa.seasonal import seasonal_decompose

result = seasonal_decompose(ts, model='additive', period=7)
result.plot()
```

**Stationarity:** A time series is stationary if its mean and variance are constant over time. Most forecasting models (ARIMA) require stationarity. Test with Augmented Dickey-Fuller (ADF) test — p-value < 0.05 → stationary.

> 💡 **Key Rule:** Apply differencing (`.diff()`) to make non-stationary series stationary. First-order difference removes linear trends. Seasonal difference removes annual patterns.

---
## 🟢 Exercise 19.1 — Decomposition and Stationarity Test *(Level 1 · Est. 15 min)*

> Decompose the time series (period=7). Plot components. Run ADF test on original and differenced series.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 19.1: Decomposition + ADF Test
# ─────────────────────────────────────────────────────────────

# YOUR CODE HERE — seasonal_decompose with period=7, plot

# ADF test on original series
# YOUR CODE HERE

# Apply first-order differencing, ADF test again
# YOUR CODE HERE

print('ADF test results:')
# YOUR CODE HERE — print p-values

In [ ]:
# @title ✅ Solution — Exercise 19.1 (click to expand)

# Decompose
result = seasonal_decompose(ts, model='additive', period=7)
fig = result.plot()
fig.set_size_inches(14, 10)
plt.suptitle('Seasonal Decomposition (period=7 days)', y=1.02)
plt.tight_layout()
plt.show()

# ADF test helper
def adf_test(series, name):
    result = adfuller(series.dropna())
    print(f'{name}: ADF statistic={result[0]:.4f}, p-value={result[1]:.6f}', end=' ')
    print('→ STATIONARY' if result[1] < 0.05 else '→ NON-STATIONARY')

adf_test(ts, 'Original')
adf_test(ts.diff(), 'First-order diff')
print('\n✅ Original series may have trend (non-stationary). Differencing often makes it stationary.')

---
## 🟡 Exercise 19.2 — ARIMA Forecasting *(Level 2 · Est. 20 min)*

> Split: train = first 2.5 years, test = last 6 months. Fit ARIMA(1,1,1). Forecast on test. Plot actual vs forecast. Compute MAE and RMSE.

In [ ]:
# @title ✅ Solution — Exercise 19.2 (click to expand)

split_date = '2023-07-01'
train_ts = ts[:split_date]
test_ts  = ts[split_date:]
print(f'Train: {len(train_ts)} days, Test: {len(test_ts)} days')

model_arima = ARIMA(train_ts, order=(1, 1, 1))
result_arima = model_arima.fit()

forecast = result_arima.forecast(steps=len(test_ts))
forecast.index = test_ts.index

mae  = mean_absolute_error(test_ts, forecast)
rmse = mean_squared_error(test_ts, forecast, squared=False)

plt.figure(figsize=(14, 5))
train_ts[-90:].plot(color='steelblue', label='Train (last 90d)')
test_ts.plot(color='darkorange', label='Actual')
forecast.plot(color='red', linestyle='--', label='ARIMA Forecast')
plt.title('ARIMA(1,1,1) Forecast vs Actual')
plt.legend()
plt.ylabel('Orders')
plt.show()

print(f'MAE:  {mae:.1f} orders/day')
print(f'RMSE: {rmse:.1f} orders/day')
print('\n✅ ARIMA captures trend but not complex seasonality. SARIMA or Prophet handle seasonality better.')

---
## 🔴 Exercise 19.3 — ML-Based Forecasting with Lag Features *(Level 3 · Est. 20 min)*

> Create lag features (lag 1, 7, 14, 28), rolling mean/std (window=7), day-of-week and month features. Train Linear Regression on train. Compare MAE vs ARIMA.

In [ ]:
# @title ✅ Solution — Exercise 19.3 (click to expand)

df_ts = pd.DataFrame({'orders': ts})

# Lag features
for lag in [1, 7, 14, 28]:
    df_ts[f'lag_{lag}'] = df_ts['orders'].shift(lag)

# Rolling statistics
df_ts['rolling_mean_7'] = df_ts['orders'].shift(1).rolling(7).mean()
df_ts['rolling_std_7']  = df_ts['orders'].shift(1).rolling(7).std()

# Calendar features
df_ts['dayofweek'] = df_ts.index.dayofweek
df_ts['month']     = df_ts.index.month

df_ts.dropna(inplace=True)

features = [c for c in df_ts.columns if c != 'orders']
X_ml = df_ts[features]
y_ml = df_ts['orders']

split_idx = df_ts.index.get_loc(split_date)
X_tr_ml, X_te_ml = X_ml.iloc[:split_idx], X_ml.iloc[split_idx:]
y_tr_ml, y_te_ml = y_ml.iloc[:split_idx], y_ml.iloc[split_idx:]

lr = LinearRegression()
lr.fit(X_tr_ml, y_tr_ml)
y_pred_ml = lr.predict(X_te_ml)

mae_ml  = mean_absolute_error(y_te_ml, y_pred_ml)
rmse_ml = mean_squared_error(y_te_ml, y_pred_ml, squared=False)

plt.figure(figsize=(14, 5))
plt.plot(y_te_ml.index, y_te_ml.values, color='darkorange', label='Actual', alpha=0.7)
plt.plot(y_te_ml.index, y_pred_ml,       color='purple',    linestyle='--', label='ML Forecast')
plt.title('ML (Lag Features + LR) Forecast vs Actual')
plt.legend()
plt.show()

print('Comparison:')
print(f'ARIMA(1,1,1) — MAE: {mae:.1f}, RMSE: {rmse:.1f}')
print(f'ML Lag + LR  — MAE: {mae_ml:.1f}, RMSE: {rmse_ml:.1f}')
print('\n✅ ML approach with lag features often beats ARIMA on complex time series.')
print('Gradient Boosting + lag features ("tabularised" time series) is popular in industry.')

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: What is stationarity and why does it matter for time series models?</strong></summary>

**Answer:** A stationary time series has constant mean, variance, and autocorrelation structure over time. ARIMA and similar models assume stationarity because they model the statistical properties of the series — if those properties change over time, the model parameters become meaningless. Non-stationarity usually comes from trends or seasonality. Differencing (subtract previous value) removes linear trend. Log transformation stabilises variance. ADF test checks stationarity statistically.
</details>

<details>
<summary><strong>Q2: What are lag features and why are they powerful for ML-based time series?</strong></summary>

**Answer:** A lag feature is the value of the series k steps back. Lag 1 = yesterday's value, lag 7 = same day last week. By including multiple lags, you let the model learn temporal dependencies — without the strict mathematical assumptions of ARIMA. You can then use any ML model (LR, GBM, Neural Network). This 'tabularisation' of time series is popular because: (1) handles non-linear patterns, (2) easy to add external features (promotions, weather, festivals), (3) parallelises well.
</details>

<details>
<summary><strong>Q3: Why should you never use random train/test split for time series?</strong></summary>

**Answer:** Time series data has temporal ordering — future data cannot be known when making past predictions. Random splitting allows the model to use 'future' data to predict 'past' values, giving artificially high performance (data leakage). Always use temporal splitting: train on early period, test on later period. For cross-validation, use TimeSeriesSplit — each fold trains on data before the test window, never after.
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 19 Complete!</h3>
<ul>
<li>Seasonal decomposition: trend, seasonality, residual</li>
<li>Stationarity test (ADF) and differencing</li>
<li>ARIMA forecasting</li>
<li>ML-based forecasting with lag and calendar features</li>
</ul>
<p><strong>Next:</strong> Chapter 20 — Career Guide and Next Steps</p>
</div>